# Clase 3 · Estadística y Python aplicado

**Caso:** Mercado Sur, una organización ficticia que vende por web y en tienda.
**Pregunta de la clase:** ¿la experiencia del canal web empeoró, o cambió la composición de clientes y operaciones observadas?

Este notebook tiene tres partes y se recorre **en orden**: antes de cada celda hay una explicación breve de qué hace y qué mirar.

1. **Repaso estadístico, con código.** Repasamos las medidas del módulo 3 corriéndolas sobre los datos, así no hace falta haber leído la teoría.
2. **Base de Python y Pandas.** Lo mínimo para leer y escribir código de análisis.
3. **Práctica guiada: el perfil del dataset.** Revisamos la calidad de los datos antes de afirmar nada.

La **actividad de grupo** está en otro notebook (`clase_3_desafio_alumnos.ipynb`). Los datos son ficticios y reproducibles.

## Paso 0 · Instalación

Corré esta celda primero. Si falta alguna librería, la instala en el mismo Python que ejecuta este notebook; si ya está todo, no hace nada.

In [ ]:
# Instalación autocontenida: instala, en el mismo Python que ejecuta este notebook, lo que falte.
import importlib, importlib.util, subprocess, sys

REQUISITOS = {"numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn", "matplotlib": "matplotlib", "jinja2": "jinja2"}
INSTALAR_XGBOOST = False          # opcional: ponelo en True para sumar XGBoost a la comparación de modelos
if INSTALAR_XGBOOST: REQUISITOS["xgboost"] = "xgboost"

print(f"Kernel en uso: {sys.executable} (Python {sys.version.split()[0]})")
faltan = [pip for modulo, pip in REQUISITOS.items() if importlib.util.find_spec(modulo) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    base = [sys.executable, "-m", "pip", "install", "-q"]
    try:
        try:
            subprocess.check_call(base + faltan)
        except subprocess.CalledProcessError:       # el Python del sistema suele rechazar instalaciones
            print("Reintentando con --break-system-packages ...")
            subprocess.check_call(base + ["--break-system-packages"] + faltan)
    except Exception:
        raise SystemExit("No se pudo instalar en este kernel. Elegí el kernel .venv (arriba a la derecha) y volvé a correr esta celda.")
    importlib.invalidate_caches()
    print("Listo.")
else:
    print("Todo instalado: podés seguir.")


## Preparación

In [ ]:
# Este archivo es autocontenido: instala solo lo que le falte (se corre una sola vez)
import importlib.util
import subprocess
import sys

LIBRERIAS = ("numpy", "pandas", "matplotlib")
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
if faltan:
    print("Instalando:", ", ".join(faltan), "(puede tardar un minuto)")
    for extra in ([], ["--user"], ["--break-system-packages"]):          # si un intento falla, prueba el siguiente
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *faltan], capture_output=True, text=True)
        if r.returncode == 0:
            break
    else:
        print("⚠️ No se pudo instalar solo. Corré en una celda:  %pip install " + " ".join(faltan))
        print(r.stderr[-300:])
    importlib.invalidate_caches()
faltan = [m for m in LIBRERIAS if importlib.util.find_spec(m) is None]
print("✅ Librerías listas" if not faltan else f"⚠️ Todavía faltan: {faltan}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

VIOLETA, VERDE, NARANJA, GRIS = "#7C5CF6", "#10D9A0", "#F59E0B", "#8888A8"
plt.rcParams.update({"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "font.size": 11, "axes.titleweight": "bold"})
pd.options.display.float_format = "{:,.2f}".format
print("pandas:", pd.__version__, "· numpy:", np.__version__)

### Armamos los datos de Mercado Sur

No cargamos ningún archivo: **los generamos con código**, con la misma semilla de las clases anteriores. Así todos tenemos exactamente los mismos datos. Son **tres tablas relacionadas**:

```
clientes ──(id_cliente)──▶ ventas ◀──(id_producto)── productos
```

La celda siguiente arma las tres tablas con un sorteo reproducible. **No hace falta leerla ahora**: lo que importa es lo que descubrimos analizando los datos.

In [ ]:
SEMILLA = 2025                                 # el «número de sorteo»
rng = np.random.default_rng(SEMILLA)           # el generador de sorteos
N_CLIENTES = 1500
N_OPS = 12000                                  # operaciones que se sortean (algunas se descartan más adelante)
print("Listo: vamos a generar", N_CLIENTES, "clientes y", N_OPS, "operaciones")
CATEGORIAS = {"Electrónica": (150, 900, 0.15), "Hogar": (30, 300, 0.20), "Indumentaria": (20, 150, 0.20),
              "Deportes": (25, 250, 0.15), "Alimentos": (3, 40, 0.30)}   # (precio mínimo, precio máximo, peso en las ventas)
filas, n = [], 1
for cat, (lo, hi, _) in CATEGORIAS.items():
    for _ in range(8):
        filas.append({"id_producto": f"P{n:03d}", "categoria": cat, "nombre": f"{cat} {n:02d}",
                      "precio_lista": round(float(rng.uniform(lo, hi)), 2)})
        n += 1
productos = pd.DataFrame(filas)
productos.head()
REGIONES = ["Centro", "Norte", "Sur", "Litoral", "Cuyo"]
clientes = pd.DataFrame({
    "id_cliente": [f"C{i:04d}" for i in range(1, N_CLIENTES + 1)],
    "region": rng.choice(REGIONES, size=N_CLIENTES, p=[0.30, 0.20, 0.18, 0.17, 0.15]),
    "fecha_alta": pd.to_datetime("2023-01-01") + pd.to_timedelta(rng.integers(0, 900, N_CLIENTES), unit="D"),
})
clientes["fecha_alta"] = clientes["fecha_alta"].dt.strftime("%Y-%m-%d")
clientes.head()
fechas = pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 365, N_OPS), unit="D")
ventas = pd.DataFrame({
    "id_venta": [f"V{i:06d}" for i in range(1, N_OPS + 1)],
    "fecha": fechas,
    "id_cliente": rng.choice(clientes["id_cliente"], size=N_OPS),
    "canal": rng.choice(["web", "tienda"], size=N_OPS, p=[0.55, 0.45]),
})
pesos = np.array([v[2] for v in CATEGORIAS.values()])
cats = np.array(list(CATEGORIAS.keys()))[rng.choice(len(CATEGORIAS), size=N_OPS, p=pesos / pesos.sum())]
ventas["id_producto"] = [rng.choice(productos.loc[productos["categoria"] == c, "id_producto"]) for c in cats]
ventas = ventas.merge(productos[["id_producto", "categoria", "precio_lista"]], on="id_producto", how="left")
ventas = ventas.merge(clientes[["id_cliente", "region"]], on="id_cliente", how="left")
ventas["unidades"] = np.where(ventas["categoria"] == "Alimentos", rng.integers(1, 9, N_OPS), rng.integers(1, 5, N_OPS))
ventas["precio_unitario"] = (ventas["precio_lista"] * rng.uniform(0.97, 1.05, N_OPS)).round(2)
ventas["descuento_pct"] = rng.choice([0.0, 0.05, 0.10, 0.15, 0.20], size=N_OPS, p=[0.45, 0.15, 0.20, 0.12, 0.08])
print(f"{len(ventas):,} operaciones sorteadas")
s2 = ventas["fecha"].dt.month > 6
seg = (ventas["canal"] == "web") & (ventas["region"] == "Litoral") & (ventas["categoria"] == "Electrónica") & s2
seg_indum = (ventas["canal"] == "web") & (ventas["categoria"] == "Indumentaria") & s2
p_comp, p_dev, p_can = np.full(N_OPS, 0.88), np.full(N_OPS, 0.07), np.full(N_OPS, 0.05)
p_comp[seg], p_dev[seg], p_can[seg] = 0.64, 0.30, 0.06
p_comp[seg_indum], p_dev[seg_indum], p_can[seg_indum] = 0.81, 0.13, 0.06
u = rng.random(N_OPS)
ventas["estado"] = np.where(u < p_comp, "completada", np.where(u < p_comp + p_dev, "devuelta", "cancelada"))
ventas = ventas.loc[~(seg & (rng.random(N_OPS) < 0.38))].copy()
print(f"Estados asignados · operaciones: {len(ventas):,}")
n_v = len(ventas)
sat = rng.choice([1, 2, 3, 4, 5], size=n_v, p=[0.05, 0.08, 0.17, 0.35, 0.35]).astype(float)
dev = (ventas["estado"] == "devuelta").to_numpy()
can = (ventas["estado"] == "cancelada").to_numpy()
sat[dev] = rng.choice([1, 2, 3], size=dev.sum(), p=[0.5, 0.3, 0.2])               # las devueltas califican peor
sat[rng.random(n_v) < np.where(can, 0.80, np.where(dev, 0.25, 0.12))] = np.nan     # y los faltantes
ventas["satisfaccion"] = sat

ventas = ventas.sort_values("fecha").reset_index(drop=True)                          # ordenadas por fecha
ventas["id_venta"] = [f"V{i:06d}" for i in range(1, len(ventas) + 1)]
ventas["fecha"] = ventas["fecha"].dt.strftime("%Y-%m-%d")
ventas = ventas[["id_venta", "fecha", "id_cliente", "id_producto", "canal", "unidades",
                 "precio_unitario", "descuento_pct", "estado", "satisfaccion"]]       # descartamos las columnas auxiliares
ventas.head()

print(f"Listo · clientes: {len(clientes):,} · productos: {len(productos):,} · ventas: {len(ventas):,}")

Dos columnas más que vamos a necesitar durante toda la clase (las explicamos en la Parte 2):

- `fecha` pasa de texto a **fecha de verdad**.
- `neto` es la facturación de cada operación: `unidades × precio_unitario × (1 − descuento_pct)`.

Y una **regla común** de Mercado Sur: la facturación realizada considera solo las operaciones **completadas**. Armamos esa tabla aparte.

In [ ]:
ventas["fecha"] = pd.to_datetime(ventas["fecha"])
ventas["neto"] = ventas["unidades"] * ventas["precio_unitario"] * (1 - ventas["descuento_pct"])
completadas = ventas[ventas["estado"] == "completada"].copy()

print(f"{len(completadas):,} operaciones completadas de {len(ventas):,}")

---
# Parte 1 · Repaso estadístico con código

Antes de comparar semestres hay que saber **cómo describir una columna de números**. Usamos el `neto` de las operaciones completadas, que es el **ticket** de cada operación.

## 1.1 · Medidas de tendencia central: ¿qué es un ticket «típico»?

Hay tres formas de resumir una columna en un solo número:

- **Media:** la suma dividida por la cantidad. Todos los valores pesan, incluso los muy grandes.
- **Mediana:** el valor del medio cuando se ordenan. La mitad de las operaciones está por debajo y la otra mitad por encima.
- **Moda:** el valor que más se repite.

**Predicción antes de correr:** ¿la media va a ser mayor, menor o igual que la mediana?

In [ ]:
neto = completadas["neto"]
centrales = pd.DataFrame({
    "medida": ["Media del ticket", "Mediana del ticket", "Moda de unidades por operación"],
    "valor": [neto.mean(), neto.median(), completadas["unidades"].mode()[0]],
})
display(centrales)

fig, ax = plt.subplots()
ax.hist(neto, bins=60, color=VIOLETA, alpha=0.85)
ax.axvline(neto.mean(), color=NARANJA, lw=2.5, label=f"Media: {neto.mean():,.0f}")
ax.axvline(neto.median(), color=VERDE, lw=2.5, label=f"Mediana: {neto.median():,.0f}")
ax.set_xlabel("Ticket (neto de la operación)"); ax.set_ylabel("Operaciones"); ax.set_title("La media queda lejos de la mediana"); ax.legend()
plt.show()

**Cómo leer la tabla y el gráfico:** la media es **416,7** y la mediana **194,5**: la media es más del doble. La mayoría de las operaciones es chica, pero unas pocas muy grandes **arrastran la media hacia la derecha**. Cuando la distribución es así, la mediana describe mejor al ticket «típico». La moda de unidades es 1: lo más frecuente es comprar una sola unidad.

## 1.2 · Medidas de dispersión: ¿cuánto varía?

Dos columnas con la misma media pueden ser muy distintas. La **dispersión** dice cuánto se alejan los valores del centro:

- **Rango:** máximo menos mínimo. Sensible a los extremos.
- **Desvío estándar:** distancia típica a la media.
- **Coeficiente de variación:** desvío ÷ media. Permite comparar dispersión entre columnas de distinta escala.
- **Cuartiles y rango intercuartílico (IQR):** el 50 % del medio de los datos, entre Q1 y Q3. No lo distorsionan los extremos.

In [ ]:
q1, q3 = neto.quantile([0.25, 0.75])
dispersion = pd.DataFrame({
    "medida": ["Mínimo", "Máximo", "Rango", "Desvío estándar", "Coeficiente de variación",
               "Q1 (25 %)", "Q3 (75 %)", "IQR (Q3 − Q1)", "Percentil 90", "Percentil 99"],
    "valor": [neto.min(), neto.max(), neto.max() - neto.min(), neto.std(), neto.std() / neto.mean(),
              q1, q3, q3 - q1, neto.quantile(0.90), neto.quantile(0.99)],
})
dispersion

**Cómo leer la tabla:** el ticket va de **10,6 a 3.751,4**. El desvío (624,1) es **1,5 veces la media**: una dispersión muy alta. El 50 % del medio de las operaciones está entre **96,5 y 404,3** (IQR de 307,7), mientras que el 1 % más caro supera los **3.197**. Dos números, dos historias: el rango dice «hay de todo» y el IQR dice «lo normal está acá».

## 1.3 · Distribuciones: la forma de los datos

La **forma** de la distribución decide qué medida conviene. Una **asimetría** positiva alta significa una cola larga hacia los valores grandes. Un **valor atípico** es el que queda fuera de `Q3 + 1,5 × IQR` (la regla de la caja del boxplot).

**Predicción antes de correr:** ¿los valores atípicos son errores de carga o tienen una explicación?

In [ ]:
limite = q3 + 1.5 * (q3 - q1)
atipicos = completadas[completadas["neto"] > limite]
print(f"Asimetría: {neto.skew():.2f} (0 = simétrica; más de 1 = cola larga a la derecha)")
print(f"Límite de atípicos: {limite:,.1f} · atípicos: {len(atipicos):,} ({len(atipicos) / len(completadas):.1%})")

por_categoria = completadas.merge(productos[["id_producto", "categoria"]], on="id_producto")
orden = por_categoria.groupby("categoria")["neto"].median().sort_values().index.tolist()
fig, ax = plt.subplots()
ax.boxplot([por_categoria.loc[por_categoria["categoria"] == c, "neto"] for c in orden],
           patch_artist=True, boxprops={"facecolor": VIOLETA, "alpha": 0.5}, medianprops={"color": NARANJA, "lw": 2})
ax.set_xticks(range(1, len(orden) + 1), orden)
ax.set_ylabel("Ticket (neto de la operación)"); ax.set_title("Los atípicos tienen explicación: es la categoría")
plt.show()

**Cómo leer el resultado:** la asimetría es **2,99**: cola larga a la derecha. Hay **1.137 atípicos (10,8 %)**, pero **no son errores**: el **87 %** son de **Electrónica**, una categoría cara por naturaleza (mediana de 1.445 contra 98 en Alimentos). Un atípico se investiga antes de borrarlo.

## 1.4 · Medidas por grupo (y la trampa de promediar promedios)

Para comparar grupos se calcula la medida **dentro de cada grupo**. Cuidado con una trampa: el promedio de los promedios de los grupos **no es** el promedio general, porque los grupos tienen tamaños distintos. Siempre hay que reportar la **cantidad** (`n`) detrás de cada número.

**Predicción antes de correr:** si promediamos las medias de las cinco categorías, ¿obtenemos la media general de 416,7?

In [ ]:
por_cat = por_categoria.groupby("categoria")["neto"].agg(n="count", media="mean", mediana="median")
display(por_cat)

print(f"Media general real:                  {neto.mean():,.1f}")
print(f"Promedio de las 5 medias (trampa):   {por_cat['media'].mean():,.1f}")

por_canal = completadas.groupby("canal")["neto"].agg(n="count", media="mean", mediana="median")
display(por_canal)

**Cómo leer las tablas:** el promedio de las cinco medias da **504,9**, no 416,7: hay 3.139 operaciones de Alimentos (baratas) y solo 1.545 de Electrónica (caras), y promediar las medias les da a todas el mismo peso. La media general ponderada es la correcta. Por canal, el año completo no muestra diferencias: la web y la tienda tienen un ticket medio casi idéntico (416,8 y 416,6). **Para ver cambios hay que mirar dentro del tiempo**, y eso es lo que hace el desafío.

## 1.5 · Faltantes: el denominador importa

La `satisfaccion` puede faltar. Un faltante **no es un cero**: es una respuesta que no existe. Lo que decidamos hacer con los faltantes cambia el **denominador** del indicador. Por regla de Mercado Sur, **se analizan y no se reemplazan automáticamente**.

**Predicción antes de correr:** ¿los faltantes se reparten parejo entre los estados de la operación?

In [ ]:
faltante = ventas["satisfaccion"].isna()
print(f"Faltantes de satisfacción: {faltante.sum():,} de {len(ventas):,} ({faltante.mean():.1%})")

por_estado = ventas.groupby("estado")["satisfaccion"].agg(operaciones="size", con_respuesta="count")
por_estado["% faltante"] = 100 * (1 - por_estado["con_respuesta"] / por_estado["operaciones"])
display(por_estado)

opciones = pd.DataFrame({
    "cómo se calcula": ["Ignorando los faltantes (lo que hace pandas)", "Solo operaciones completadas", "Reemplazando el faltante por 0 (incorrecto)"],
    "satisfacción media": [ventas["satisfaccion"].mean(), completadas["satisfaccion"].mean(), ventas["satisfaccion"].fillna(0).mean()],
})
opciones

**Cómo leer las tablas:** los faltantes **no son parejos**: faltan en el **78,6 % de las canceladas**, el 25,5 % de las devueltas y el 12,2 % de las completadas. Tiene sentido: quien cancela no suele calificar. Por eso la media cambia con el universo: **3,74** con todas las operaciones, **3,88** solo con las completadas y **3,12** si se reemplaza por 0, que es un resultado **falso**: inventa clientes que calificaron con cero. Reportá siempre qué universo usaste.

## 1.6 · Asociación no es causalidad

La **correlación** (entre -1 y 1) mide cuánto se mueven juntas dos columnas **en línea recta**. Una correlación alta no demuestra que una cause la otra, y una correlación cercana a 0 solo dice que **no hay relación lineal**.

In [ ]:
columnas = ["unidades", "precio_unitario", "descuento_pct", "neto", "satisfaccion"]
corr = completadas[columnas].corr()
display(corr)

fig, ax = plt.subplots(figsize=(5.5, 4.4))
im = ax.imshow(corr, cmap="PuOr_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(columnas)), columnas, rotation=40, ha="right"); ax.set_yticks(range(len(columnas)), columnas)
for i in range(len(columnas)):
    for j in range(len(columnas)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=9)
ax.grid(False); ax.set_title("Correlación entre columnas"); plt.colorbar(im); plt.show()

**Cómo leer la matriz:** el `neto` se mueve con el `precio_unitario` (**0,86**, fuerte), lo cual es esperable porque el neto se calcula con el precio. En cambio la `satisfaccion` no tiene relación lineal con nada (todas cercanas a 0): ni el descuento ni el monto explican cómo califican. Que no haya correlación no significa que no haya relación: puede ser **no lineal** o estar en un subgrupo. Lo veremos en el desafío.

---
# Parte 2 · Base de Python y Pandas

Lo mínimo para **leer y escribir** el código de análisis. Primero Python solo; después, la librería Pandas, que es la que usamos para trabajar con tablas.

## 2.1 · Python mínimo

Variables y tipos de datos: cada valor tiene un **tipo**, y el tipo decide qué se puede hacer con él.

In [ ]:
nombre = "Mercado Sur"      # texto (str)
clientes_n = 1500           # número entero (int)
descuento = 0.15            # número con decimales (float)
activo = True               # verdadero o falso (bool)

for valor in (nombre, clientes_n, descuento, activo):
    print(f"{valor!r:>15}  →  {type(valor).__name__}")

**Listas y diccionarios.** Una **lista** guarda valores en orden; un **diccionario** guarda pares `clave: valor` y se consulta por la clave. Los **condicionales** (`if`) y los **bucles** (`for`) deciden y repiten.

In [ ]:
canales = ["web", "tienda"]                                   # lista
precios = {"Electrónica": 900, "Hogar": 300, "Alimentos": 40}  # diccionario
precios["Deportes"] = 250                                     # se agrega un par

for categoria, precio in precios.items():                     # bucle sobre el diccionario
    etiqueta = "cara" if precio > 200 else "barata"           # condicional
    print(f"{categoria}: {precio} → {etiqueta}")
print("Canales:", canales, "· cantidad:", len(canales))

**Funciones.** Una función **guarda un cálculo con nombre** para reutilizarlo. Recibe datos, devuelve un resultado. Es la forma de no repetir código, y de que el cálculo esté en un solo lugar.

In [ ]:
def neto_operacion(unidades, precio_unitario, descuento_pct):
    """Facturación neta de una operación: unidades × precio × (1 − descuento)."""
    return unidades * precio_unitario * (1 - descuento_pct)

print(neto_operacion(3, 100, 0.10))        # 3 × 100 × 0,90 = 270

## 2.2 · De Python a Pandas: la tabla como objeto

Un **DataFrame** es una tabla; cada **columna** es una `Serie`. La gran ventaja: una operación sobre una columna **se aplica a todas las filas de una vez**, sin escribir un bucle.

**Predicción antes de correr:** ¿el cálculo con bucle y el vectorizado dan lo mismo? ¿Cuál es más corto?

In [ ]:
primeras = ventas.head(5)

con_bucle = []                                           # a mano, fila por fila
for _, fila in primeras.iterrows():
    con_bucle.append(neto_operacion(fila["unidades"], fila["precio_unitario"], fila["descuento_pct"]))

vectorizado = primeras["unidades"] * primeras["precio_unitario"] * (1 - primeras["descuento_pct"])   # toda la columna junta

print("Con bucle:   ", [round(x, 2) for x in con_bucle])
print("Vectorizado: ", [round(x, 2) for x in vectorizado])

Mismo resultado, y el vectorizado es una línea y es mucho más rápido en tablas grandes. **Para el análisis, pensá en columnas completas, no en filas.**

## 2.3 · Explorar un DataFrame

Lo primero con cualquier tabla: **qué forma tiene, qué tipos tiene cada columna y cómo se ve**.

In [ ]:
print("Forma (filas, columnas):", ventas.shape)
print("\nTipos por columna:")
print(ventas.dtypes.to_string())
ventas.head()

In [ ]:
print("Valores de canal:", ventas["canal"].unique())
display(ventas["estado"].value_counts().to_frame("operaciones").assign(porcentaje=lambda t: 100 * t["operaciones"] / t["operaciones"].sum()))
ventas.describe().T[["count", "mean", "std", "min", "50%", "max"]]

**Cómo leer `describe()`:** muestra, por cada columna numérica, la cantidad de datos (`count`), la media, el desvío, el mínimo, la **mediana** (`50%`) y el máximo. Dos cosas para mirar siempre: si `count` es menor que el total (hay faltantes: acá, la `satisfaccion` tiene 9.999 de 11.969) y si el mínimo y el máximo son razonables.

## 2.4 · Seleccionar y filtrar

Para **filtrar filas** se arma una condición que da verdadero o falso por fila (una *máscara*) y se usa dentro de corchetes. Se combinan con `&` (y) y `|` (o), siempre entre paréntesis.

In [ ]:
web = ventas[ventas["canal"] == "web"]
web_electronica = ventas.merge(productos[["id_producto", "categoria"]], on="id_producto")
web_electronica = web_electronica[(web_electronica["canal"] == "web") & (web_electronica["categoria"] == "Electrónica")]
sur_o_cuyo = clientes[clientes["region"].isin(["Sur", "Cuyo"])]

print(f"Operaciones web: {len(web):,}")
print(f"Operaciones web de Electrónica: {len(web_electronica):,}")
print(f"Clientes del Sur o de Cuyo: {len(sur_o_cuyo):,}")
ventas.loc[ventas["unidades"] >= 7, ["id_venta", "canal", "unidades", "neto"]].head()

## 2.5 · Limpieza: duplicados, valores imposibles y faltantes

Antes de analizar, tres controles. **Predicción:** ¿cuántas filas perdemos si usamos `dropna()` sobre toda la tabla?

In [ ]:
print("id_venta repetidos:      ", ventas["id_venta"].duplicated().sum())
print("Filas totalmente iguales:", ventas.duplicated().sum())
print("Descuento fuera de 0–1:  ", ((ventas["descuento_pct"] < 0) | (ventas["descuento_pct"] > 1)).sum())
print("Unidades menores o iguales a 0:", (ventas["unidades"] <= 0).sum())
print(f"Descuento igual a 0: {(ventas['descuento_pct'] == 0).mean():.1%}  ← un 0 es un dato válido, no un faltante")

In [ ]:
sin_na = ventas.dropna()
perdidas = ventas.loc[~ventas.index.isin(sin_na.index)]
print(f"dropna() deja {len(sin_na):,} filas y descarta {len(perdidas):,} ({len(perdidas) / len(ventas):.1%})")

comparacion = pd.DataFrame({"antes": ventas["estado"].value_counts(), "después de dropna()": sin_na["estado"].value_counts()})
comparacion["se pierde"] = (1 - comparacion["después de dropna()"] / comparacion["antes"]).map("{:.1%}".format)
comparacion

**Cómo leer los controles:** no hay duplicados ni valores imposibles. El descuento es 0 en el **45,5 %** de las operaciones: un cero es un dato válido (sin descuento), **no un faltante**. Y `dropna()` descarta **1.970 filas (16,5 %)**, pero no de forma pareja: elimina el **79 %** de las operaciones canceladas, así que el análisis quedaría **sesgado** hacia las que salieron bien. Moraleja: el faltante se **analiza**, no se borra a ciegas.

## 2.6 · Transformar: columnas nuevas, unir tablas y agrupar

Las tres transformaciones más usadas: **crear columnas**, **unir tablas** (`merge`) y **agrupar y resumir** (`groupby`). Al unir, hay que comprobar que **no cambie la cantidad de filas**: es el control del *grano*.

In [ ]:
ventas["mes"] = ventas["fecha"].dt.month
ventas["semestre"] = np.where(ventas["mes"] <= 6, "S1", "S2")

base = (ventas.merge(productos[["id_producto", "categoria"]], on="id_producto", how="left")
              .merge(clientes[["id_cliente", "region"]], on="id_cliente", how="left"))

assert len(base) == len(ventas), "¡El merge duplicó o perdió filas!"           # control del grano
print(f"Filas antes de unir: {len(ventas):,} · después: {len(base):,} → el grano se mantiene")
base[["id_venta", "fecha", "semestre", "canal", "categoria", "region", "neto"]].head()

In [ ]:
resumen = (base[base["estado"] == "completada"]
           .groupby(["categoria", "semestre"])["neto"]
           .agg(operaciones="count", facturacion="sum", mediana="median")
           .round(1))
display(resumen)

pivote = base[base["estado"] == "completada"].pivot_table(index="categoria", columns="canal", values="neto", aggfunc="sum").round(0)
pivote.sort_values("web", ascending=False)

**Cómo leer las tablas:** `groupby` arma un grupo por cada combinación de categoría y semestre y calcula varias medidas a la vez. `pivot_table` reacomoda el resultado en una grilla (categorías en filas, canales en columnas), más fácil de leer. La facturación usa solo operaciones **completadas**, por la regla común.

## 2.7 · Visualización básica

La regla: **el tipo de gráfico lo decide la pregunta**. Comparar categorías: barras. Ver evolución en el tiempo: líneas. Ver la forma de una distribución: histograma.

In [ ]:
completas = base[base["estado"] == "completada"]
fig, ejes = plt.subplots(1, 2, figsize=(13, 4.2))

mediana_cat = completas.groupby("categoria")["neto"].median().sort_values()
ejes[0].barh(mediana_cat.index, mediana_cat.values, color=VIOLETA)
ejes[0].set_title("Barras: comparar categorías"); ejes[0].set_xlabel("Ticket mediano")

por_mes = completas.groupby(["mes", "canal"]).size().unstack()
for canal, color in zip(por_mes.columns, [NARANJA, VIOLETA]):
    ejes[1].plot(por_mes.index, por_mes[canal], marker="o", lw=2.5, color=color, label=canal)
ejes[1].set_title("Líneas: evolución en el tiempo"); ejes[1].set_xlabel("Mes"); ejes[1].set_ylabel("Operaciones completadas"); ejes[1].legend()
plt.tight_layout(); plt.show()

Un buen gráfico tiene **título que diga qué mostrar**, ejes con nombre y **una sola idea**. Antes de graficar, siempre se validan los tipos y el universo.

---
# Parte 3 · Práctica guiada: el perfil del dataset

**Consigna:** crear un **perfil del dataset**: filas, claves, duplicados, faltantes, rango temporal y distribución de estados. Para **cada hallazgo**, proponer una corrección o decisión explícita.

El perfil es lo primero que hace un analista con datos nuevos. Evita afirmar cosas sobre datos que no se revisaron.

## 3.1 · Las tres tablas: filas y claves

In [ ]:
perfil_tablas = pd.DataFrame({
    "tabla": ["clientes", "productos", "ventas"],
    "una fila representa": ["un cliente", "un producto", "una operación de venta"],
    "filas": [len(clientes), len(productos), len(ventas)],
    "clave": ["id_cliente", "id_producto", "id_venta"],
    "clave única": [clientes["id_cliente"].is_unique, productos["id_producto"].is_unique, ventas["id_venta"].is_unique],
})
perfil_tablas

## 3.2 · Integridad de las claves, duplicados y rango temporal

In [ ]:
integridad = pd.Series({
    "Ventas con id_cliente que no existe en clientes": int((~ventas["id_cliente"].isin(clientes["id_cliente"])).sum()),
    "Ventas con id_producto que no existe en productos": int((~ventas["id_producto"].isin(productos["id_producto"])).sum()),
    "Clientes sin ninguna operación": int((~clientes["id_cliente"].isin(ventas["id_cliente"])).sum()),
    "Filas totalmente duplicadas en ventas": int(ventas.duplicated().sum()),
})
display(integridad.to_frame("cantidad"))
print(f"Rango temporal de ventas: {ventas['fecha'].min().date()} → {ventas['fecha'].max().date()}")
print(f"Alta de clientes: {pd.to_datetime(clientes['fecha_alta']).min().date()} → {pd.to_datetime(clientes['fecha_alta']).max().date()}")

## 3.3 · Faltantes y distribución de estados

In [ ]:
faltantes = ventas.isna().sum().to_frame("faltantes")
faltantes["% del total"] = 100 * faltantes["faltantes"] / len(ventas)
display(faltantes[faltantes["faltantes"] > 0])

estados = ventas["estado"].value_counts().to_frame("operaciones")
estados["% del total"] = 100 * estados["operaciones"] / len(ventas)
estados

## 3.4 · Los cinco controles

Esta función junta **cinco controles** en un reporte. Es el modelo de lo que cada equipo tiene que mostrar en su desafío: cada control declara **qué verifica** y si **pasó**.

In [ ]:
def cinco_controles(v, c, p):
    controles = {
        "1 · Grano: id_venta es único (una fila = una operación)": v["id_venta"].is_unique,
        "2 · Claves: todas las ventas tienen cliente y producto": bool(v["id_cliente"].isin(c["id_cliente"]).all() and v["id_producto"].isin(p["id_producto"]).all()),
        "3 · Fechas: todas dentro de 2025": bool(v["fecha"].between("2025-01-01", "2025-12-31").all()),
        "4 · Estados: solo completada, devuelta o cancelada": bool(v["estado"].isin(["completada", "devuelta", "cancelada"]).all()),
        "5 · Valores: descuento entre 0 y 1 y unidades positivas": bool(v["descuento_pct"].between(0, 1).all() and (v["unidades"] > 0).all()),
    }
    for descripcion, ok in controles.items():
        print("✔" if ok else "✘", descripcion)
    return all(controles.values())

todo_ok = cinco_controles(ventas, clientes, productos)
print("\nPerfil limpio:", todo_ok)

## 3.5 · Hallazgo → decisión

Cada hallazgo del perfil necesita una **decisión explícita**: qué hacemos y por qué. Completen la tabla en equipo (editen esta celda de texto):

| Hallazgo | Decisión explícita | Por qué |
| --- | --- | --- |
| La `satisfaccion` falta en el 16,5 % de las operaciones (78,6 % de las canceladas) | *(escriban acá)* | *(escriban acá)* |
| Hay un cliente sin operaciones | *(escriban acá)* | *(escriban acá)* |
| El descuento es 0 en el 45,5 % de las filas | *(escriban acá)* | *(escriban acá)* |
| Los tres estados conviven en la misma tabla | *(escriban acá)* | *(escriban acá)* |
| Las tablas no tienen duplicados ni claves rotas | *(escriban acá)* | *(escriban acá)* |

---
## Para llevarte

- **Media y mediana** cuentan cosas distintas: con colas largas, la mediana describe mejor lo típico.
- **Dispersión** y **forma** de la distribución van siempre junto a la medida central.
- **Promediar promedios** da un número falso: reportá siempre el `n` de cada grupo.
- Un **faltante no es un cero** y `dropna()` indiscriminado sesga: se analiza, no se borra.
- Pensá en **columnas completas**, no en filas; y al unir tablas, controlá el **grano**.
- Antes de afirmar algo, un **perfil del dataset** y cinco controles.
- Una correlación **no demuestra causalidad**.

**Evidencia de salida del equipo:** notebook reproducible con cinco controles, una visualización, un hallazgo y una limitación.